In [ ]:
from pathlib import Path
import mlflow, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').resolve().as_posix()}")
runs = mlflow.search_runs(experiment_names=["intrusion_baseline"])
if runs.empty:
    raise RuntimeError("No intrusion_baseline runs found. Run ml.train_intrusion first.")
runs["split"] = runs["params.split"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# chart 1: same model, three splits
s = (runs[runs["params.with_port"] == "False"]
     .groupby("split")["metrics.attack_recall"].max()
     .reindex(["random", "block", "unseen"]))
print(s)                                   # check the numbers before trusting the chart
s.plot.bar(ax=ax[0], color=["#c44", "#e90", "#27a"], rot=0)
ax[0].set_title("Attack recall by split"); ax[0].set_ylim(0, 1.05)

# chart 2: leave-one-family-out
d = pd.read_csv(ROOT / "evaluation/results/leave_one_family_out.csv").set_index("held_out")
print(d[["n_attack_test", "xgb_recall_at_validation_1pct_fpr", "iso_recall_at_validation_1pct_fpr"]].round(3))
d[["xgb_recall_at_validation_1pct_fpr", "iso_recall_at_validation_1pct_fpr"]].plot.bar(ax=ax[1], rot=30)
ax[1].set_title("Recall on never-seen family (threshold calibrated to 1% validation FPR)"); ax[1].set_ylim(0, 1.05)

fig.tight_layout()
out = ROOT / "evaluation/results/leakage_and_generalization.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150)
plt.show()

# chart 1: same model, three splits (attack recall is what changes)

In [ ]:
s = runs[runs["params.with_port"] == "False"].groupby("split")["metrics.attack_recall"].max()
s.reindex(["random", "block", "unseen"]).plot.bar(ax=ax[0], color=["#c44", "#e90", "#27a"])
ax[0].set_title("Attack recall by split"); ax[0].set_ylim(0, 1.05)

# chart 2: leave-one-family-out

In [ ]:
d = pd.read_csv("../evaluation/results/leave_one_family_out.csv").set_index("held_out")
d[["xgb_recall_at_validation_1pct_fpr", "iso_recall_at_validation_1pct_fpr"]].plot.bar(ax=ax[1])
ax[1].set_title("Recall on never-seen family (threshold calibrated to 1% validation FPR)")
plt.tight_layout(); plt.savefig("../evaluation/results/leakage_and_generalization.png", dpi=150)